In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
import time
from tqdm import tqdm
import torch.nn.functional as F


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"
TILE_SIZE  = 256       
STRIDE     = 256 

SHOW_PROGRESS = False
batch_size = 16
num_epochs =50
learning_rate = 1e-4


print(f"Tile Size     : {TILE_SIZE}×{TILE_SIZE}")

print(f"Device        : {device}")
print(f"Stride        : {STRIDE}")
print(f"Batch Size    : {batch_size}")
print(f"Epochs        : {num_epochs}")
print(f"Learning Rate : {learning_rate}")

Tile Size     : 256×256
Device        : cuda
Stride        : 256
Batch Size    : 16
Epochs        : 50
Learning Rate : 0.0001


In [3]:
class PatchedChangeDataset(Dataset):
    def __init__(self, root_dir, augment=False,
                 tile_size=TILE_SIZE, stride=STRIDE,
                 min_change_pixels=10,
                 neg_ratio=1.5):          # None = keep ALL patches (val/test)
        self.root_dir  = root_dir
        self.augment   = augment
        self.tile_size = tile_size
        self.stride    = stride

        self.A = sorted(os.listdir(os.path.join(root_dir, "A")))
        self.B = sorted(os.listdir(os.path.join(root_dir, "B")))
        self.Y = sorted(os.listdir(os.path.join(root_dir, "label")))

        _probe = Image.open(os.path.join(root_dir, "A", self.A[0]))
        full_W, full_H = _probe.size
        _probe.close()

        raw_patches = []
        if full_H == tile_size and full_W == tile_size:
            for i in range(len(self.A)):
                raw_patches.append((i, 0, 0))
        else:
            for i in range(len(self.A)):
                for r in range(0, full_H - tile_size + 1, stride):
                    for c in range(0, full_W - tile_size + 1, stride):
                        raw_patches.append((i, r, c))

        split_name = os.path.basename(root_dir)

        # ── neg_ratio=None → val/test: keep ALL patches, no filtering ──
        if neg_ratio is None:
            self.patches = raw_patches
            print(f"  [{split_name}] {len(self.A)} images → "
                  f"{len(self.patches)} patches (NO filtering — full distribution)")
        else:
            # ── Train: controlled negative sampling ──
            print(f"  [{split_name}] Scanning {len(raw_patches)} patches...", end=" ")

            change_patches    = []
            no_change_patches = []

            _mask_cache_idx = -1
            _mask_cache_arr = None

            for (img_idx, r, c) in raw_patches:
                if img_idx != _mask_cache_idx:
                    mask_path = os.path.join(root_dir, "label", self.Y[img_idx])
                    _mask_cache_arr = np.array(Image.open(mask_path).convert("L"))
                    _mask_cache_idx = img_idx

                tile_mask = _mask_cache_arr[r:r + tile_size, c:c + tile_size]
                n_change  = (tile_mask > 127).sum()

                if n_change >= min_change_pixels:
                    change_patches.append((img_idx, r, c))
                else:
                    no_change_patches.append((img_idx, r, c))

            # REFINEMENT 3: seed before shuffle for reproducibility
            np.random.seed(42)
            np.random.shuffle(no_change_patches)

            # REFINEMENT 1: neg_ratio=1.5 (not 1.0) for realistic distribution
            max_neg  = int(len(change_patches) * neg_ratio)
            kept_neg = no_change_patches[:max_neg]

            self.patches = change_patches + kept_neg
            np.random.seed(42)
            np.random.shuffle(self.patches)

            print(f"done")
            print(f"  [{split_name}] Change patches : {len(change_patches)}")
            print(f"  [{split_name}] No-change kept : {len(kept_neg)} "
                  f"(ratio={neg_ratio}× → max {max_neg})")
            print(f"  [{split_name}] Total used     : {len(self.patches)} "
                  f"/ {len(raw_patches)} raw\n")

        self.to_tensor_img  = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225])
        ])
        self.to_tensor_mask = transforms.ToTensor()

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_idx, r, c = self.patches[idx]
        box = (c, r, c + self.tile_size, r + self.tile_size)

        img1 = Image.open(os.path.join(self.root_dir, "A", self.A[img_idx])).convert("RGB").crop(box)
        img2 = Image.open(os.path.join(self.root_dir, "B", self.B[img_idx])).convert("RGB").crop(box)
        mask = Image.open(os.path.join(self.root_dir, "label", self.Y[img_idx])).convert("L").crop(box)

        if self.augment:
            if torch.rand(1) < 0.5:
                img1 = TF.hflip(img1); img2 = TF.hflip(img2); mask = TF.hflip(mask)
            if torch.rand(1) < 0.5:
                img1 = TF.vflip(img1); img2 = TF.vflip(img2); mask = TF.vflip(mask)
            if torch.rand(1) < 0.3:
                angle = transforms.RandomRotation.get_params([-30, 30])
                img1 = TF.rotate(img1, angle, interpolation=InterpolationMode.BILINEAR)
                img2 = TF.rotate(img2, angle, interpolation=InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
            if torch.rand(1) < 0.4:
                img1 = TF.adjust_brightness(img1, torch.empty(1).uniform_(0.75, 1.25).item())
                img1 = TF.adjust_contrast(img1,   torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_brightness(img2, torch.empty(1).uniform_(0.75, 1.25).item())
                img2 = TF.adjust_contrast(img2,   torch.empty(1).uniform_(0.75, 1.25).item())

        img1 = self.to_tensor_img(img1)
        img2 = self.to_tensor_img(img2)
        mask = self.to_tensor_mask(mask)
        mask = (mask > 0.5).float()

        return img1, img2, mask

In [4]:
ROOT = "/kaggle/input/datasets/mdrifaturrahman33/levir-cd/LEVIR CD"

print("Building patch datasets...")
train_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "train"), augment=True,
    min_change_pixels=10,
    neg_ratio=1.5          # REFINEMENT 1: 1.5× negatives → realistic urban distribution
)
val_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "val"), augment=False,
    neg_ratio=None          # REFINEMENT 2: full distribution, no filtering
)
test_dataset = PatchedChangeDataset(
    os.path.join(ROOT, "test"), augment=False,
    neg_ratio=None          # REFINEMENT 2: full distribution, no filtering
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches/epoch : {len(train_loader)}")
print(f"Val   batches/epoch : {len(val_loader)}")
print(f"Test  batches       : {len(test_loader)}")

Building patch datasets...
  [train] Scanning 7120 patches... done
  [train] Change patches : 3153
  [train] No-change kept : 3967 (ratio=1.5× → max 4729)
  [train] Total used     : 7120 / 7120 raw

  [val] 64 images → 1024 patches (NO filtering — full distribution)
  [test] 128 images → 2048 patches (NO filtering — full distribution)
Train batches/epoch : 445
Val   batches/epoch : 64
Test  batches       : 128


In [5]:
# Run after Cell 4 — check that patches now have change pixels
change_ratios = [train_dataset[i][2].mean().item() for i in range(20)]
print("Sample change ratios (first 20 patches):")
print([f"{r:.4f}" for r in change_ratios])
# Expect: mix of 0.0000 (kept neg) and 0.02–0.40 (change patches)
# NOT all 0.0000 as before

Sample change ratios (first 20 patches):
['0.1643', '0.0251', '0.0092', '0.0389', '0.0000', '0.0000', '0.0000', '0.0000', '0.2831', '0.0177', '0.0000', '0.0000', '0.0000', '0.0000', '0.0000', '0.0000', '0.0109', '0.0375', '0.0000', '0.0556']


In [6]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: recalibrates channel importance"""
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // reduction, ch, 1, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)


class DecoderBlock(nn.Module):
    """
    Upsample → concat RICH skip [f1, f2, |f1-f2|] → Conv-BN-ReLU → SE
    Rich skip triples channels but preserves individual T1/T2 context
    SE then recalibrates which channels matter most for change
    """
    def __init__(self, in_ch, out_ch, skip_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
        self.se = SEBlock(out_ch)   # ← Priority 6: SE after each decoder block

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)   # skip = cat[f1, f2, |f1-f2|]
        return self.se(self.conv(x))

In [7]:
class ChangeDetectionNet(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        self.stage0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.stage1 = nn.Sequential(backbone.maxpool, backbone.layer1)
        self.stage2 = backbone.layer2
        self.stage3 = backbone.layer3
        self.stage4 = backbone.layer4

        self.cross_attn4 = nn.MultiheadAttention(512, 8, batch_first=True, dropout=0.1)
        self.cross_attn3 = nn.MultiheadAttention(256, 8, batch_first=True, dropout=0.1)

        # Priority 5: skip_ch = 3× original because skip = [f1, f2, |f1-f2|]
        self.dec4 = DecoderBlock(512, 256, skip_ch=256*3)   # diff3: 256×3=768
        self.dec3 = DecoderBlock(256, 128, skip_ch=128*3)   # s2:    128×3=384
        self.dec2 = DecoderBlock(128,  64, skip_ch= 64*3)   # s1:     64×3=192
        self.dec1 = DecoderBlock( 64,  64, skip_ch= 64*3)   # s0:     64×3=192

        self.head = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1)
        )

        # Priority 7: auxiliary heads for deep supervision
        self.aux_dec4 = nn.Conv2d(256, 1, 1)   # after dec4 → 16×16
        self.aux_dec3 = nn.Conv2d(128, 1, 1)   # after dec3 → 32×32

    def encode(self, x):
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)
        return s0, s1, s2, s3, s4

    def _cross_attn(self, attn_module, f1_feat, f2_feat):
        B, C, H, W = f1_feat.shape
        q = f1_feat.flatten(2).permute(0, 2, 1)
        k = f2_feat.flatten(2).permute(0, 2, 1)
        out, _ = attn_module(q, k, k)
        return out.permute(0, 2, 1).reshape(B, C, H, W)

    def _rich_skip(self, a, b):
        """Concat [f1, f2, |f1-f2|] — preserves temporal direction + magnitude"""
        return torch.cat([a, b, torch.abs(a - b)], dim=1)

    def forward(self, x1, x2):
        f1 = self.encode(x1)
        f2 = self.encode(x2)

        # Multi-scale attention
        attn4  = self._cross_attn(self.cross_attn4, f1[4], f2[4])
        diff4  = torch.abs(f1[4] - attn4)

        attn3  = self._cross_attn(self.cross_attn3, f1[3], f2[3])
        diff3  = self._rich_skip(f1[3], f2[3])   # rich skip replaces raw diff3

        # Decoder with rich concat skips
        d4 = self.dec4(diff4, diff3)                          # [B,256,16,16]
        d3 = self.dec3(d4,    self._rich_skip(f1[2], f2[2]))  # [B,128,32,32]
        d2 = self.dec2(d3,    self._rich_skip(f1[1], f2[1]))  # [B, 64,64,64]
        d1 = self.dec1(d2,    self._rich_skip(f1[0], f2[0]))  # [B, 64,128,128]

        # Main output
        p1 = self.head(d1)   # [B,1,256,256]

        # Priority 7: auxiliary outputs (only used during training)
        aux4 = self.aux_dec4(d4)   # [B,1,16,16]
        aux3 = self.aux_dec3(d3)   # [B,1,32,32]

        return p1, aux4, aux3

In [8]:
torch.manual_seed(42)
np.random.seed(42)

model = ChangeDetectionNet().to(device)

_a = torch.randn(2, 3, 256, 256).to(device)
_b = torch.randn(2, 3, 256, 256).to(device)

_out, _aux4, _aux3 = model(_a, _b)   # ← unpack tuple

print(f"Main output shape : {_out.shape}")    # Expect [2, 1, 256, 256]
print(f"Aux4 output shape : {_aux4.shape}")   # Expect [2, 1,  16,  16]
print(f"Aux3 output shape : {_aux3.shape}")   # Expect [2, 1,  32,  32]
print(f"Output max        : {_out.max().item():.4f}")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params  : {total_params/1e6:.2f}M")   # Expect ~14.83M

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


Main output shape : torch.Size([2, 1, 256, 256])
Aux4 output shape : torch.Size([2, 1, 16, 16])
Aux3 output shape : torch.Size([2, 1, 32, 32])
Output max        : 0.1363
Trainable params  : 16.46M


loss fn

In [9]:

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce  = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

POS_WEIGHT = 2.5
bce   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(device))
focal = FocalLoss(alpha=0.25, gamma=2.0)

def dice_loss(pred, target, smooth=1e-6):
    pred   = torch.sigmoid(pred)
    pred   = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def criterion(pred, target):
    # Focal handles hard edge pixels, BCE handles class balance, Dice handles region overlap
    return bce(pred, target) + dice_loss(pred, target) + 0.5 * focal(pred, target)

In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, weight_decay=1e-4
)

# Restarts at epoch 20, 60, 140 — escapes plateaus periodically
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6
)

Stage 1 — Dual Temporal Input
Input Image T1
Input Image T2

Stage 2 — Shared Siamese Encoder (ResNet18 Backbone)
T1 → ResNet18 Encoder → F1
T2 → ResNet18 Encoder → F2

Output:
F1 = [1,512,8,8]
F2 = [1,512,8,8]

This stage corresponds to Siamese deep feature extraction used in early Siamese change detection papers and urban Siamese frameworks

Stage 3 — Tokenization for Transformer Attention
F1 → flatten → tokens
F2 → flatten → tokens

Shape:
[1,512,8,8] → [1,64,512]

Stage 4 — Cross Attention Fusion
Q = F1
K = F2
V = F2

Cross-attention:
Attention(F1,F2)
Output:
attn_out = [1,64,512]

Stage 5 — Attention Map Reconstruction
[1,64,512] → [1,512,8,8]

Result:
attn_map

Stage 6 — Difference Generation
(Current implied stage)
diff = |F1 - attn_map|
This is your current feature difference map.
Output:
diff = [1,512,8,8]


Stage 7 — Decoder Level 1
512 → 256
8×8 → 16×16

Output:
[1,256,16,16]


Stage 8 — Decoder Level 2
256 → 128
16×16 → 32×32
Output:
[1,128,32,32]
Stage 9 — Decoder Level 3
128 → 64
32×32 → 64×64
Stage 10 — Decoder Level 4
64 → 32
64×64 → 128×128
Stage 11 — Decoder Level 5 (Final Pixel Change Map)
32 → 1
128×128 → 256×256

Final:
Change Map = [1,1,256,256]

Stage 12 — Sigmoid Binary Pixel Probability
0 = no change
1 = change
Output values:
0 to 1 probability map
Full Current Architecture in One Line
T1,T2
→ Siamese Encoder
→ Cross Attention
→ Difference Map
→ Decoder Stack
→ Pixel Change Map
Current Architecture Position Relative to Full Research Architecture

You have completed:
✅ Siamese backbone
✅ Transformer fusion
✅ Pixel decoder

In [11]:
def compute_metrics(pred, target):
    pred   = torch.sigmoid(pred)
    pred   = (pred > 0.5).float()
    tp = (pred  *  target      ).sum().float()
    tn = ((1-pred) * (1-target)).sum().float()
    fp = (pred  * (1-target)   ).sum().float()
    fn = ((1-pred) *  target   ).sum().float()
    return tp, tn, fp, fn

def finalize_metrics(tp, tn, fp, fn):
    eps  = 1e-8
    acc  = (tp + tn) / (tp + tn + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1   = 2 * prec * rec / (prec + rec + eps)
    iou  = tp / (tp + fp + fn + eps)   # FIX: Added IoU — required for paper comparison
    return {
        "acc" : acc.item(),
        "prec": prec.item(),
        "rec" : rec.item(),
        "f1"  : f1.item(),
        "iou" : iou.item()
    }

In [12]:
history = {
    "train_loss": [], "val_loss" : [],
    "train_acc" : [], "val_acc"  : [],
    "train_prec": [], "val_prec" : [],
    "train_rec" : [], "val_rec"  : [],
    "train_f1"  : [], "val_f1"   : [],
    "train_iou" : [], "val_iou"  : [],
}
best_val_f1 = 0.0
print("Training Started\n")
print(
    f"{'Ep':>3} | {'Time':>6} | "
    f"{'TrLoss':>7} | {'TrAcc':>6} | {'TrPrec':>7} | {'TrRec':>6} | {'TrF1':>6} | {'TrIoU':>7} | "
    f"{'VlLoss':>7} | {'VlAcc':>6} | {'VlPrec':>7} | {'VlRec':>6} | {'VlF1':>6} | {'VlIoU':>7} | {'LR':>9}"
)
print("─" * 140)

for epoch in range(num_epochs):
    start_time = time.time()

    # ──────────────── TRAIN ────────────────
    model.train()
    train_loss = 0.0
    tr_tp = tr_tn = tr_fp = tr_fn = 0

    for img1, img2, mask in tqdm(train_loader, leave=False, disable=not SHOW_PROGRESS):
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)

        # ── Deep supervision: model returns main + 2 aux outputs ──
        pred, aux4, aux3 = model(img1, img2)

        # Downsample mask to match aux output sizes
        mask_16 = F.interpolate(mask, size=aux4.shape[-2:], mode='nearest')
        mask_32 = F.interpolate(mask, size=aux3.shape[-2:], mode='nearest')

        # Main loss + weighted auxiliary losses
        loss = criterion(pred, mask) \
             + 0.4 * criterion(aux4, mask_16) \
             + 0.2 * criterion(aux3, mask_32)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        # Metrics computed on main output only (pred), not aux
        tp, tn, fp, fn = compute_metrics(pred.detach(), mask)
        tr_tp += tp; tr_tn += tn; tr_fp += fp; tr_fn += fn

    # ──────────────── VALIDATION ────────────────
    model.eval()
    val_loss = 0.0
    v_tp = v_tn = v_fp = v_fn = 0

    with torch.no_grad():
        for img1, img2, mask in val_loader:
            img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
            # Discard aux outputs during validation — only eval main prediction
            pred, _, _ = model(img1, img2)
            val_loss += criterion(pred, mask).item()
            tp, tn, fp, fn = compute_metrics(pred, mask)
            v_tp += tp; v_tn += tn; v_fp += fp; v_fn += fn

    # ──────────────── FINALIZE ────────────────
    tr_m = finalize_metrics(tr_tp, tr_tn, tr_fp, tr_fn)
    val_m = finalize_metrics(v_tp, v_tn, v_fp, v_fn)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # ──────────────── LOG TO HISTORY ────────────────
    history["train_loss"].append(train_loss / len(train_loader))
    history["val_loss"  ].append(val_loss   / len(val_loader))
    history["train_acc" ].append(tr_m["acc"])
    history["val_acc"   ].append(val_m["acc"])
    history["train_prec"].append(tr_m["prec"])
    history["val_prec"  ].append(val_m["prec"])
    history["train_rec" ].append(tr_m["rec"])
    history["val_rec"   ].append(val_m["rec"])
    history["train_f1"  ].append(tr_m["f1"])
    history["val_f1"    ].append(val_m["f1"])
    history["train_iou" ].append(tr_m["iou"])
    history["val_iou"   ].append(val_m["iou"])

    # ──────────────── SAVE BEST ────────────────
    if val_m["f1"] > best_val_f1:
        best_val_f1 = val_m["f1"]
        torch.save(model.state_dict(), "best_model_stage1_fixed.pth")
        saved_marker = " ✓"
    else:
        saved_marker = ""

    m, s = divmod(int(time.time() - start_time), 60)

    # ──────────────── PRINT ALL METRICS ────────────────
    print(
        f"{epoch+1:3d} | {m:2d}m {s:2d}s | "
        f"{history['train_loss'][-1]:7.4f} | "
        f"{tr_m['acc']:6.4f} | {tr_m['prec']:7.4f} | {tr_m['rec']:6.4f} | "
        f"{tr_m['f1']:6.4f} | {tr_m['iou']:7.4f} | "
        f"{history['val_loss'][-1]:7.4f} | "
        f"{val_m['acc']:6.4f} | {val_m['prec']:7.4f} | {val_m['rec']:6.4f} | "
        f"{val_m['f1']:6.4f} | {val_m['iou']:7.4f} | "
        f"{current_lr:.3e}"
        f"{saved_marker}"
    )

    if (epoch + 1) % 10 == 0:
        print("─" * 140)

print(f"\nBest Val F1 : {best_val_f1:.4f}")

Training Started

 Ep |   Time |  TrLoss |  TrAcc |  TrPrec |  TrRec |   TrF1 |   TrIoU |  VlLoss |  VlAcc |  VlPrec |  VlRec |   VlF1 |   VlIoU |        LR
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  1 |  6m 14s |  1.8356 | 0.9731 |  0.6730 | 0.7744 | 0.7201 |  0.5627 |  0.9719 | 0.9686 |  0.5756 | 0.9542 | 0.7181 |  0.5601 | 9.939e-05 ✓
  2 |  6m  3s |  0.7440 | 0.9841 |  0.7855 | 0.8834 | 0.8316 |  0.7118 |  0.4832 | 0.9874 |  0.8191 | 0.8987 | 0.8571 |  0.7499 | 9.758e-05 ✓
  3 |  6m 38s |  0.4232 | 0.9882 |  0.8582 | 0.8816 | 0.8697 |  0.7695 |  0.4243 | 0.9894 |  0.8703 | 0.8775 | 0.8739 |  0.7760 | 9.460e-05 ✓
  4 |  8m 19s |  0.3488 | 0.9895 |  0.8738 | 0.8944 | 0.8839 |  0.7920 |  0.3904 | 0.9899 |  0.8660 | 0.8991 | 0.8823 |  0.7893 | 9.055e-05 ✓
  5 |  6m 33s |  0.3230 | 0.9901 |  0.8811 | 0.8990 | 0.8900 |  0.8017 |  0.3776 | 0.9909 |  0.9186 | 0.8599 | 0.8882 |  0.7990 | 8.55

In [13]:
model.load_state_dict(torch.load("best_model_stage1_fixed.pth"))
model.eval()

test_loss = 0.0
t_tp = t_tn = t_fp = t_fn = 0

with torch.no_grad():
    for img1, img2, mask in test_loader:
        img1, img2, mask = img1.to(device), img2.to(device), mask.to(device)
        pred, _, _ = model(img1, img2)   # discard aux4, aux3
        test_loss += criterion(pred, mask).item()
        tp, tn, fp, fn = compute_metrics(pred, mask)
        t_tp += tp; t_tn += tn; t_fp += fp; t_fn += fn

test_m = finalize_metrics(t_tp, t_tn, t_fp, t_fn)

print("\n" + "═" * 50)
print("  TEST RESULTS — code1-9")
print("═" * 50)
print(f"  Loss      : {test_loss / len(test_loader):.4f}")
print("─" * 50)
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['prec']:.4f}")
print(f"  Recall    : {test_m['rec']:.4f}")
print(f"  F1 Score  : {test_m['f1']:.4f}")
print(f"  IoU       : {test_m['iou']:.4f}")
print("═" * 50)


══════════════════════════════════════════════════
  TEST RESULTS — code1-9
══════════════════════════════════════════════════
  Loss      : 0.3133
──────────────────────────────────────────────────
  Accuracy  : 0.9907
  Precision : 0.9175
  Recall    : 0.8975
  F1 Score  : 0.9074
  IoU       : 0.8304
══════════════════════════════════════════════════
